In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
product_data = pd.read_excel("validation_lookup_comparision.xlsx")
category_data=pd.read_excel("full unspsc list.xlsx")

In [ ]:
nlp = spacy.load("en_core_web_lg")

In [ ]:
def clean_product_description(text):
    units = [
        'MM', 'CM', 'M', 'L', 'SWG', 'SQMM', 'HP', 'W', 'V', 'A', 'DC',
        'KG', 'G', 'SQ', 'CUB', 'IN', 'FT', 'VDC', 'NO', 'NC','CF'
    ]
    
    patterns = [
        # Prefix patterns (MM: and PN:)
        r'(?:MM:|PN:)[^,]*(?:,|\s|$)',
        
        # Product codes with letters, numbers, and special characters
        r'(?:[A-Z]+[-]?\d+[A-Z0-9-]+)',  # Matches ACF-100N4-IRBP, VSP-25
        
        # Service specifications
        r'SERVICE:[A-Z]+',  # Matches SERVICE:WATER    ---------- double check ------------
        
        # Complex product codes
        r'PVM\d+[A-Z0-9]+',  # Matches PVM141ER10GS02AAA28000000A0A
        r'VERDERAIR-VA-\d+',  # Matches VERDERAIR-VA-25
        r'KDS-\d+\+?',  # Matches KDS-1537+
        
        # Measurements and dimensions
        rf'\b\d+(?:\.\d+)?(?:\s)?(?:{"|".join(units)})\b',  # Basic measurements with units
        r'\b\d+(?:\.\d+)?(?:\s)?(?:x|\*)\s?\d+(?:\.\d+)?(?:\s)?(?:x|\*)\s?\d+(?:\.\d+)?\b',  # 3D dimensions
        r'\b\d+(?:\.\d+)?(?:\s)?(?:x|\*)\s?\d+(?:\.\d+)?\b',  # 2D dimensions
        
        # Specific formats
        r'MM:DSM-?\d+/\d+',  # Matches MM:DSM-125/40
        r'[A-Z]{2,3}\d+/\d+',  # Matches RKB125/27
        r'\d+ST\b',  # Matches 6ST
        r'\d+(?:NO|NC)(?:\+\d+(?:NO|NC))?\s+\d+V(?:DC)?',  # Matches 2NO+2NC 110VDC
        r'P\s*NO\.\s*\d+',  # Matches P NO.190
        r'UP\s+\d+/\d+',  # Matches UP 100/38
        r'UP\d+/\d+[A-Z]*,.*$',
        r',?\s*UP\d+/\d+[A-Z]*(?:,.*)?$'
        
        # Additional measurements
        r'M\d+\*\d+(?:\.\d+)?\s*SQMM\s*\d+\.\d+M',  # Matches M3*1 SQMM 0.08M
        r'\d+\.\d+\s*(?:CON)?\s*\d+[WP]\s*\d+[WP]\s*\d+CM\s*\d+/\d+',  # Matches 2.54 CON 2W 2P 15CM 7/36
        
        # Product specifications
        r'HYD,[A-Z0-9]+',  # Matches HYD,PVM...
        r'\b\d+/\d+\b',  # Fractions and ratios
        r'L\d+(?:\.\d+)?[A-Z]*',  # Length specifications
        r'\b\d+[WP]\b',  # Matches 2W, 2P
        r'[A-Z]\d+\*\d+(?:\.\d+)?',  # Size with units at start
    ]
    

    cleaned = text
    for pattern in patterns:
        matches = re.finditer(pattern, cleaned, re.IGNORECASE)
        for match in matches:
            removed_patterns.append(match.group())
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r'\s+', ' ', cleaned)  # Replace multiple spaces with single space
    cleaned = re.sub(r',\s*,', ',', cleaned)  # Remove double commas
    cleaned = re.sub(r'^\s*,|\s*,\s*$', '', cleaned)  # Remove leading/trailing commas
    cleaned = cleaned.strip()
    cleaned = re.sub(r'\(\s*\)', '', cleaned)  # Remove empty parentheses
    cleaned= re.sub(r'[^a-zA-Z0-9\s]', ' ', cleaned)
    cleaned=re.sub(r'[^a-zA-Z\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()  # Remove extra spaces
    cleaned=cleaned.lower()
    return cleaned

In [ ]:
clean_product_description("HYD PUMP ACF-100N4-IRBP (IMO PUMPS)")

In [ ]:
product_data.columns

In [ ]:
product_data['Cleaned Material Description']=product_data['Material description'].apply(clean_product_description)

In [ ]:
category_data['Level 4 Description']=category_data['Level 4 Description'].apply(clean_product_description)

In [ ]:
tfidf=TfidfVectorizer(binary=True,use_idf=False, norm=None)

In [ ]:
tfidf.fit(category_data['Level 4 Description'])

In [ ]:
product_vectors = tfidf.transform(product_data['Cleaned Material Description'])
category_vectors = tfidf.transform(category_data['Level 4 Description'])

In [ ]:
similarity_matrix = cosine_similarity(product_vectors, category_vectors)

In [ ]:
similarity_matrix.shape

In [ ]:
model= SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
refined_mapped_categories = []

for index, row in product_data.iterrows():
    similarity_scores = similarity_matrix[index if index < len(similarity_matrix) else len(similarity_matrix) - 1]
    top_indices = similarity_scores.argsort()[-100:][::-1]

    top_100_categories= category_data['Level 4 Description'].iloc[top_indices].tolist()

    product_embedding = model.encode(row['Cleaned Material Description'])
    category_embeddings = model.encode(top_100_categories)

    refined_similarity=model.similarity(product_embedding,category_embeddings)[0]
    refined_similarity_array=np.array(refined_similarity)

    top_5_indices = refined_similarity_array.argsort()[-5:][::-1]

    top_5_categories=[]
    for idx in top_5_indices:
        category = top_100_categories[idx]
        top_5_categories.append(category)

    refined_mapped_categories.append(", ".join(top_5_categories))

product_data['Mapped Category'] = refined_mapped_categories

In [ ]:
product_data.to_excel('Cleaned_results.xlsx', index=False)